# Fashion MNIST — Bayesian Optimization with KerasTuner
This final example tunes hidden layer count, layer width, dropout, learning rate, and **batch size**. Unlike the previous Random Search example, Bayesian optimization uses earlier trial results to suggest new settings.

[Original example](https://www.kaggle.com/code/mahsazamanifard/fashion-mnist-dataset-hp-tuning-using-keras-tuner)

## 1. Imports

In [ ]:
# %pip install --upgrade tensorflow keras-tuner scikit-learn matplotlib
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(7)


## 2. Prepare training, validation, and test data
Use 10,000 training images and 2,000 separate validation images for a quick demonstration. The test set is used only at the end. Increase the sample counts for a larger experiment.

Pixels are normalized once in the model using `Rescaling`, so keep the input images in their original 0–255 range.

In [ ]:
(x, y), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train, x_val, y_train, y_val = train_test_split(
    x, y, train_size=10000, test_size=2000,
    stratify=y, random_state=7
)
print('Training:', x_train.shape)
print('Validation:', x_val.shape)
print('Test:', x_test.shape)

plt.imshow(x_train[10], cmap='gray')
plt.title(f'Class: {y_train[10]}')
plt.axis('off')
plt.show()


## 3. Define the model and tunable parameters

| Parameter | Values | Effect |
|---|---|---|
| `num_hidden_layers` | 1, 2, 3 | More layers increase depth and computation; they can also make training harder. |
| `num_units` | 8, 16, 32 | More units increase capacity, but may increase overfitting. Each hidden layer uses the same width. |
| `dropout_rate` | 0.1–0.5, step 0.1 | Drops a fraction of hidden units during training. Too much can cause underfitting. |
| `learning_rate` | 0.0001, 0.001, 0.01 | Controls the update size. Very small values train slowly; very large values can destabilize training. |
| `batch_size` | 32, 64, 96, 128 | Controls samples per update, update frequency, and memory use. It is selected in the custom tuner below. |

Epochs are a fixed training budget, not a tuned parameter here. The output is always 10 softmax units for the 10 classes.

In [ ]:
def create_model(hp):
    num_hidden_layers = hp.Choice('num_hidden_layers', values=[1, 2, 3])
    num_units = hp.Choice('num_units', values=[8, 16, 32])
    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1)
    learning_rate = hp.Choice('learning_rate', values=[0.0001, 0.001, 0.01])

    model = keras.Sequential()
    model.add(keras.Input(shape=(28, 28)))
    model.add(keras.layers.Rescaling(1.0 / 255.0))
    model.add(keras.layers.Flatten())

    for layer in range(num_hidden_layers):
        model.add(keras.layers.Dense(num_units, activation='relu'))
        model.add(keras.layers.Dropout(dropout_rate))

    model.add(keras.layers.Dense(10, activation='softmax'))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

create_model(kt.HyperParameters()).summary()


## 4. Add batch size tuning
`batch_size` belongs to `fit`, rather than to the model architecture. The small custom tuner passes the selected value to training and lets the standard tuner handle everything else.

Register batch size before creating the tuner so it appears in the search-space summary from the start.

In [ ]:
class CustomTuner(kt.BayesianOptimization):
    def run_trial(self, trial, *args, **kwargs):
        kwargs['batch_size'] = trial.hyperparameters.get('batch_size')
        return super().run_trial(trial, *args, **kwargs)

hp = kt.HyperParameters()
hp.Int('batch_size', min_value=32, max_value=128, step=32)


## 5. Configure Bayesian optimization
The first **3 trials** provide random starting points; later trials can use the Bayesian model. Setting this explicitly matters: the default number of initial points is three times the search-space dimension, which exceeds this seven-trial demonstration.

`overwrite=True` replaces earlier results in this example's project folder. Use `False` to resume the same unchanged experiment, or change the project name to keep a separate experiment.

In [ ]:
tuner = CustomTuner(
    create_model,
    objective='val_accuracy',
    max_trials=7,
    num_initial_points=3,
    hyperparameters=hp,
    seed=7,
    directory='keras_tuner_results',
    project_name='fashion_mnist_bayesian_batch',
    overwrite=True
)
tuner.search_space_summary()


## 6. Search
Each trial trains for at most five epochs. Validation accuracy selects the trial; the test data are not passed to the search. Seven trials explore only a small part of the space.

In [ ]:
tuner.search(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=5,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor='val_accuracy', mode='max', patience=2
    )],
    verbose=2
)


## 7. Inspect the selected settings
Batch size appears in the hyperparameter results, not in `model.summary()`, because it does not change the model architecture.

In [ ]:
tuner.results_summary()
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
for name, value in best_hp.values.items():
    print(name, ':', value)


## 8. Train a fresh model using the selected settings
Build a new model with the chosen hyperparameters and use the **chosen batch size**. Early stopping selects the best validation epoch within a maximum of 20 epochs and restores its weights.

A longer training budget can change performance; these settings were selected using five-epoch trials. This demonstration does not guarantee globally optimal settings.

In [ ]:
model = create_model(best_hp)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy', mode='max',
    patience=2, min_delta=0.001, restore_best_weights=True
)
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=20,
    batch_size=best_hp.get('batch_size'),
    callbacks=[early_stopping],
    verbose=2
)


## 9. Inspect learning curves and evaluate once on the test set

In [ ]:
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.xlabel('Epoch (starting at 0)')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_accuracy:.3f}')


### Student questions
1. Why does batch size require a custom training step rather than a Dense-layer argument?
2. What is the difference between `max_trials` and `epochs`?
3. Why must the test set remain outside the tuning process?
4. What can you infer if training accuracy rises while validation accuracy stops improving?

[BayesianOptimization API](https://keras.io/keras_tuner/api/tuners/bayesian/) · [Tuner API](https://keras.io/keras_tuner/api/tuners/base_tuner/)